# LeetCode 938 — Range Sum of BST bằng Python

**Link đề:** https://leetcode.com/problems/range-sum-of-bst/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search (DFS), Binary Search Tree (BST), Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def rangeSumBST(self, root: Optional[TreeNode], low: int, high: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **cây nhị phân** (binary tree), `TreeNode`, và cách LeetCode biểu diễn cây bằng danh sách level-order (`[10,5,15,3,7,null,18]`).
2. Hiểu tính chất **cây nhị phân tìm kiếm (BST)**: trái < gốc < phải.
3. Viết **đệ quy** (recursion) DFS duyệt cây; sau đó **cắt tỉa** (pruning) nhờ tính chất BST.
4. Viết bản **lặp** dùng stack, bản BFS dùng queue, bản duyệt **inorder** dừng sớm.
5. Trace từng nút, chứng minh tính đúng, Time/Space, lỗi thường gặp và bài tập.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — tổng các nút có giá trị trong `[low, high]`

Cho gốc `root` của một **BST** và hai số `low`, `high`. Trả về **tổng giá trị của mọi nút** có giá trị nằm trong đoạn **đóng** `[low, high]`.

Ví dụ 1: `root = [10,5,15,3,7,null,18]`, `low = 7`, `high = 15` → `32` (nút 7, 10, 15).

```text
        10
       /  \
      5    15
     / \     \
    3   7     18
```

Ví dụ 2: `root = [10,5,15,3,7,13,18,1,null,6]`, `low = 6`, `high = 10` → `23` (nút 6, 7, 10).

```text
          10
        /    \
       5      15
      / \    /  \
     3   7  13   18
    /   /
   1   6
```

**Constraints:** số nút trong `[1, 2·10^4]`; `1 <= Node.val <= 10^5`; `1 <= low <= high <= 10^5`; mọi giá trị **khác nhau**.

## 2. Nền tảng Python và cấu trúc dữ liệu

### 2.1. `TreeNode` — một nút của cây

Mỗi nút có `val` (giá trị), `left` (con trái), `right` (con phải). Con không tồn tại là `None`.

Ta tự định nghĩa class để notebook chạy độc lập (LeetCode đã định nghĩa sẵn). `Optional[TreeNode]` nghĩa là "một `TreeNode` hoặc `None`".

In [1]:
from typing import Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"

# Dựng tay cây nhỏ:   2
#                    / \
#                   1   3
root = TreeNode(2, TreeNode(1), TreeNode(3))
print(root, root.left, root.right, root.left.left)

TreeNode(2) TreeNode(1) TreeNode(3) None


`__repr__` quy định cách đối tượng được in ra — giúp debug dễ hơn thay vì `<__main__.TreeNode object at 0x...>`.

### 2.2. Dựng cây từ danh sách level-order kiểu LeetCode

LeetCode ghi cây theo **từng tầng, trái sang phải**; `null` (trong Python là `None`) đánh dấu chỗ trống. Thuật toán dựng:
1. Phần tử đầu là gốc; đưa gốc vào hàng đợi.
2. Lấy lần lượt nút ra khỏi hàng đợi; hai phần tử tiếp theo của danh sách là con trái, con phải của nó (bỏ qua `None`).
3. Con mới tạo được đưa vào hàng đợi.

In [2]:
from collections import deque

def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def to_list(root: Optional[TreeNode]) -> list:
    # Chuyển ngược cây -> danh sách level-order (bỏ None thừa ở cuối)
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

t1 = build_tree([10, 5, 15, 3, 7, None, 18])
print(to_list(t1))
t2 = build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6])
print(to_list(t2))

[10, 5, 15, 3, 7, None, 18]
[10, 5, 15, 3, 7, 13, 18, 1, None, 6]


### 2.3. In cây dạng "nằm ngang" để nhìn cấu trúc

Hàm `show` in cây xoay 90° (gốc bên trái, nhánh phải ở trên). Dùng **đệ quy**: in cây con phải, in nút, in cây con trái, mỗi tầng thụt vào thêm.

In [3]:
def show(node, depth=0):
    if node is None:
        return
    show(node.right, depth + 1)
    print("      " * depth + str(node.val))
    show(node.left, depth + 1)

show(t2)

            18
      15
            13
10
            7
                  6
      5
            3
                  1


### 2.4. Đệ quy (recursion) là gì?

Một hàm **gọi lại chính nó** trên bài toán **nhỏ hơn**. Luôn cần:
1. **Trường hợp cơ sở** (base case): dừng, không gọi tiếp — với cây thường là `node is None`.
2. **Bước đệ quy**: giải bài toán cho cây con, rồi kết hợp.

Ví dụ: đếm số nút = `1 + đếm(trái) + đếm(phải)`; cây rỗng có 0 nút.

In [4]:
def count_nodes(node):
    if node is None:          # base case
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def tree_sum(node):
    if node is None:
        return 0
    return node.val + tree_sum(node.left) + tree_sum(node.right)

print("số nút t2 =", count_nodes(t2), "| tổng t2 =", tree_sum(t2))

số nút t2 = 9 | tổng t2 = 78


### 2.5. Tính chất BST

Cây nhị phân **tìm kiếm** (BST): với **mọi** nút, mọi giá trị trong cây con **trái** `<` giá trị nút `<` mọi giá trị trong cây con **phải**.

Hệ quả quan trọng: duyệt **inorder** (trái → nút → phải) cho dãy **tăng dần**.

In [5]:
def inorder(node, out):
    if node is None:
        return out
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)
    return out

print("inorder t1:", inorder(t1, []))
print("inorder t2:", inorder(t2, []))

inorder t1: [3, 5, 7, 10, 15, 18]
inorder t2: [1, 3, 5, 6, 7, 10, 13, 15, 18]


## 3. Cách trực giác — duyệt mọi nút, cộng nút nằm trong đoạn

Không cần biết BST: duyệt toàn bộ cây (DFS), nút nào có `low <= val <= high` thì cộng.

In [6]:
def range_sum_full(root: Optional[TreeNode], low: int, high: int) -> int:
    if root is None:
        return 0
    here = root.val if low <= root.val <= high else 0
    return here + range_sum_full(root.left, low, high) + range_sum_full(root.right, low, high)

print(range_sum_full(t1, 7, 15))    # 32
print(range_sum_full(t2, 6, 10))    # 23

32
23


### 3.1. Phiên bản "gom rồi lọc" (dễ đọc nhất)

Dùng `inorder` lấy mọi giá trị, rồi `sum` những giá trị trong đoạn. Đơn giản nhưng tốn thêm list `O(n)`.

In [7]:
def range_sum_collect(root, low, high):
    return sum(v for v in inorder(root, []) if low <= v <= high)

print(range_sum_collect(t1, 7, 15), range_sum_collect(t2, 6, 10))

32 23


## 4. Nút thắt và insight — cắt tỉa nhờ BST

Cách trực giác **luôn** thăm cả `n` nút, kể cả những nhánh chắc chắn không có giá trị nào trong đoạn.

**Insight (BST):** tại nút có giá trị `v`:
- Nếu `v < low`: nút này **và toàn bộ cây con trái** (đều `< v < low`) bị loại → **chỉ đi sang phải**.
- Nếu `v > high`: nút này **và toàn bộ cây con phải** bị loại → **chỉ đi sang trái**.
- Nếu `low <= v <= high`: cộng `v`, và **cả hai** phía đều có thể còn giá trị hợp lệ → đi cả hai.

Ví dụ t1 với `[7, 15]`: tại nút `5` (< 7), bỏ hẳn nút `3` mà không cần thăm. Tại `18`? Ta vẫn phải thăm 18 vì `15` nằm trong đoạn (đi cả hai phía).

In [8]:
# Đếm số nút được thăm: duyệt toàn bộ vs cắt tỉa
def visits_full(node):
    if node is None:
        return 0
    return 1 + visits_full(node.left) + visits_full(node.right)

def visits_pruned(node, low, high):
    if node is None:
        return 0
    if node.val < low:
        return 1 + visits_pruned(node.right, low, high)
    if node.val > high:
        return 1 + visits_pruned(node.left, low, high)
    return 1 + visits_pruned(node.left, low, high) + visits_pruned(node.right, low, high)

print("t1 [7,15]: full =", visits_full(t1), "| pruned =", visits_pruned(t1, 7, 15))
print("t2 [6,10]: full =", visits_full(t2), "| pruned =", visits_pruned(t2, 6, 10))

t1 [7,15]: full = 6 | pruned = 5
t2 [6,10]: full = 9 | pruned = 6


## 5. Lời giải chính — DFS đệ quy có cắt tỉa

**Mảnh 1 — base case:** `if node is None: return 0`.

**Mảnh 2 — quá nhỏ:** `if node.val < low: return dfs(node.right)`.

**Mảnh 3 — quá lớn:** `if node.val > high: return dfs(node.left)`.

**Mảnh 4 — trong đoạn:** `return node.val + dfs(node.left) + dfs(node.right)`.

Ta viết hàm con `dfs` lồng bên trong để không phải truyền `low`, `high` mỗi lần (hàm con "nhìn thấy" biến của hàm ngoài — gọi là **closure**).

In [9]:
def range_sum_pruned(root: Optional[TreeNode], low: int, high: int) -> int:
    def dfs(node):
        if node is None:
            return 0
        if node.val < low:
            return dfs(node.right)
        if node.val > high:
            return dfs(node.left)
        return node.val + dfs(node.left) + dfs(node.right)
    return dfs(root)

print(range_sum_pruned(t1, 7, 15))   # 32
print(range_sum_pruned(t2, 6, 10))   # 23

32
23


## 6. TRACE từng bước

Cây t2, `low = 6`, `high = 10`:

| Nút thăm | So sánh | Quyết định | Đóng góp |
|:---:|---|---|:---:|
| 10 | 6 ≤ 10 ≤ 10 | cộng, đi trái & phải | 10 |
| 5 | 5 < 6 | bỏ nút & cây trái (3, 1), chỉ đi phải | 0 |
| 7 | 6 ≤ 7 ≤ 10 | cộng, đi trái & phải | 7 |
| 6 | 6 ≤ 6 ≤ 10 | cộng | 6 |
| 15 | 15 > 10 | bỏ nút & cây phải (18), chỉ đi trái | 0 |
| 13 | 13 > 10 | chỉ đi trái (None) | 0 |

Tổng `10 + 7 + 6 = 23`. Các nút 3, 1, 18 **không bao giờ được thăm**.

In [10]:
def trace_range_sum(root, low, high):
    total = 0
    def dfs(node, depth):
        nonlocal total
        pad = "  " * depth
        if node is None:
            return
        if node.val < low:
            print(f"{pad}{node.val}: < {low} -> bỏ trái, chỉ đi phải")
            dfs(node.right, depth + 1)
        elif node.val > high:
            print(f"{pad}{node.val}: > {high} -> bỏ phải, chỉ đi trái")
            dfs(node.left, depth + 1)
        else:
            total += node.val
            print(f"{pad}{node.val}: trong [{low},{high}] -> cộng, total = {total}")
            dfs(node.left, depth + 1)
            dfs(node.right, depth + 1)
    dfs(root, 0)
    print("Kết quả:", total)
    return total

trace_range_sum(t2, 6, 10)
print()
trace_range_sum(t1, 7, 15)

10: trong [6,10] -> cộng, total = 10
  5: < 6 -> bỏ trái, chỉ đi phải
    7: trong [6,10] -> cộng, total = 17
      6: trong [6,10] -> cộng, total = 23
  15: > 10 -> bỏ phải, chỉ đi trái
    13: > 10 -> bỏ phải, chỉ đi trái
Kết quả: 23

10: trong [7,15] -> cộng, total = 10
  5: < 7 -> bỏ trái, chỉ đi phải
    7: trong [7,15] -> cộng, total = 17
  15: trong [7,15] -> cộng, total = 32
    18: > 15 -> bỏ phải, chỉ đi trái
Kết quả: 32


32

`nonlocal total` cho phép hàm con **gán lại** biến `total` của hàm bao ngoài (nếu không có, Python coi `total` trong `dfs` là biến cục bộ mới và báo lỗi `UnboundLocalError` khi `+=`).

## 7. Các lời giải khác và bảng so sánh

### 7.1. DFS lặp bằng stack (tránh giới hạn độ sâu đệ quy)

Python giới hạn độ sâu đệ quy khoảng 1000 tầng. Cây BST **lệch** (như danh sách móc nối) với `2·10^4` nút có thể vượt giới hạn! Bản lặp dùng **stack** tự quản lý.

In [11]:
import sys
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())

def range_sum_stack(root: Optional[TreeNode], low: int, high: int) -> int:
    total = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.val < low:
            stack.append(node.right)
        elif node.val > high:
            stack.append(node.left)
        else:
            total += node.val
            stack.append(node.left)
            stack.append(node.right)
    return total

print(range_sum_stack(t1, 7, 15), range_sum_stack(t2, 6, 10))

Giới hạn đệ quy mặc định: 1000
32 23


### 7.2. BFS bằng queue (duyệt theo tầng)

Thay stack bằng `deque` + `popleft` → thứ tự thăm theo tầng. Kết quả tổng như nhau.

In [12]:
def range_sum_bfs(root, low, high):
    total = 0
    q = deque([root])
    while q:
        node = q.popleft()
        if node is None:
            continue
        if low <= node.val <= high:
            total += node.val
        if node.val > low:          # còn có thể có giá trị hợp lệ bên trái
            q.append(node.left)
        if node.val < high:         # còn có thể có giá trị hợp lệ bên phải
            q.append(node.right)
    return total

print(range_sum_bfs(t1, 7, 15), range_sum_bfs(t2, 6, 10))

32 23


Bản này viết điều kiện cắt tỉa theo kiểu khác: chỉ đi trái nếu `val > low` (bên trái nhỏ hơn `val`, có thể vẫn `>= low`), chỉ đi phải nếu `val < high`. Tương đương bản 3 nhánh.

### 7.3. Inorder lặp, dừng sớm khi vượt `high`

Inorder cho dãy tăng → khi gặp giá trị `> high` có thể **dừng hẳn**.

In [13]:
def range_sum_inorder_stop(root, low, high):
    total = 0
    stack, node = [], root
    while stack or node:
        while node:                 # đi hết về trái
            stack.append(node)
            node = node.left
        node = stack.pop()
        if node.val > high:
            break                   # mọi giá trị sau còn lớn hơn
        if node.val >= low:
            total += node.val
        node = node.right
    return total

print(range_sum_inorder_stop(t1, 7, 15), range_sum_inorder_stop(t2, 6, 10))

32 23


### 7.4. Bảng so sánh

`n` = số nút, `h` = chiều cao cây, `k` = số nút nằm trong đoạn.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Gom inorder rồi lọc | `O(n)` | `O(n)` | dễ hiểu nhất |
| DFS toàn bộ | `O(n)` | `O(h)` | không dùng BST |
| **DFS đệ quy cắt tỉa** | `O(n)` xấu nhất, thường ≈ `O(h + k)` | `O(h)` | nên nộp |
| DFS lặp bằng stack | như trên | `O(h)`…`O(n)` | không lo giới hạn đệ quy |
| BFS | như trên | `O(chiều rộng)` | duyệt theo tầng |
| Inorder dừng sớm | `O(h + k)` | `O(h)` | khai thác thứ tự tăng |

## 8. Code nộp LeetCode

LeetCode đã có `TreeNode` và `Optional`. Bản đệ quy cắt tỉa:

In [14]:
class Solution:
    def rangeSumBST(self, root: Optional[TreeNode], low: int, high: int) -> int:
        if root is None:
            return 0
        if root.val < low:
            return self.rangeSumBST(root.right, low, high)
        if root.val > high:
            return self.rangeSumBST(root.left, low, high)
        return (root.val
                + self.rangeSumBST(root.left, low, high)
                + self.rangeSumBST(root.right, low, high))

sol = Solution()
assert sol.rangeSumBST(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15) == 32
assert sol.rangeSumBST(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 10) == 23
print("Solution PASS 2 ví dụ đề bài")

Solution PASS 2 ví dụ đề bài


Nếu lo cây lệch quá sâu, nộp bản stack (mục 7.1) — an toàn hơn trong Python:

```python
class Solution:
    def rangeSumBST(self, root, low, high):
        total, stack = 0, [root]
        while stack:
            node = stack.pop()
            if not node:
                continue
            if node.val < low:
                stack.append(node.right)
            elif node.val > high:
                stack.append(node.left)
            else:
                total += node.val
                stack.append(node.left)
                stack.append(node.right)
        return total
```

## 9. Vì sao thuật toán đúng?

Gọi `S(node)` là tổng các giá trị trong `[low, high]` thuộc **cây con** gốc `node`. Ta chứng minh `dfs(node) == S(node)` bằng **quy nạp** theo kích thước cây con.

- **Cơ sở:** cây rỗng → `S = 0` = `dfs(None)`.
- **Trường hợp `v < low`:** `v` không thuộc đoạn. Mọi giá trị cây con trái `< v < low` → đóng góp 0. Vậy `S(node) = S(node.right) = dfs(node.right)` (giả thiết quy nạp).
- **Trường hợp `v > high`:** đối xứng → `S(node) = S(node.left)`.
- **Trường hợp `low <= v <= high`:** `S(node) = v + S(left) + S(right)`.

Mỗi trường hợp khớp đúng một nhánh của code. ∎

Điểm then chốt: việc **bỏ cả cây con** chỉ hợp lệ nhờ tính chất BST — với cây nhị phân thường thì phải dùng `range_sum_full`.

In [15]:
# Phản ví dụ: cây KHÔNG phải BST -> cắt tỉa cho kết quả sai, duyệt toàn bộ vẫn đúng
not_bst = TreeNode(5, TreeNode(9), TreeNode(1))   # 9 ở bên trái 5: vi phạm BST
print("full  :", range_sum_full(not_bst, 6, 10))
print("pruned:", range_sum_pruned(not_bst, 6, 10), "<- bỏ sót 9 vì tin vào tính chất BST")

full  : 9
pruned: 0 <- bỏ sót 9 vì tin vào tính chất BST


## 10. Phân tích Time / Space

Gọi `n` = số nút, `h` = chiều cao.

- **Time:** xấu nhất `O(n)` (khi đoạn `[low, high]` bao gần hết cây, phải thăm gần mọi nút). Khi đoạn hẹp, chỉ thăm các nút trên "hai đường biên" và các nút trong đoạn: khoảng `O(h + k)`.
- **Space:** độ sâu ngăn xếp gọi hàm `O(h)`. Cây cân bằng `h ≈ log n`; cây lệch `h = n`.

Đo số nút được thăm trên cây cân bằng `n = 1023` với đoạn hẹp:

In [16]:
def build_balanced(lo, hi):
    if lo > hi:
        return None
    mid = (lo + hi) // 2
    return TreeNode(mid, build_balanced(lo, mid - 1), build_balanced(mid + 1, hi))

big = build_balanced(1, 1023)
for low, high in [(500, 510), (1, 1023), (1000, 1023)]:
    print(f"[{low},{high}]: full thăm {visits_full(big)} nút | pruned thăm {visits_pruned(big, low, high)} nút")

[500,510]: full thăm 1023 nút | pruned thăm 20 nút
[1,1023]: full thăm 1023 nút | pruned thăm 1023 nút
[1000,1023]: full thăm 1023 nút | pruned thăm 32 nút


## 11. Bộ test — mẫu đề, biên, đối chiếu

Nhóm test:
1. Hai ví dụ đề.
2. Một nút: trong / ngoài đoạn.
3. `low == high` (đúng một giá trị).
4. Đoạn bao toàn cây; đoạn không chứa nút nào.
5. Cây lệch trái / lệch phải sâu (kiểm tra bản lặp không tràn đệ quy).
6. Ngẫu nhiên: chèn giá trị ngẫu nhiên vào BST, đối chiếu với tổng lọc.

In [17]:
tests = [
    ([10, 5, 15, 3, 7, None, 18], 7, 15, 32),
    ([10, 5, 15, 3, 7, 13, 18, 1, None, 6], 6, 10, 23),
    ([5], 5, 5, 5),
    ([5], 1, 4, 0),
    ([10, 5, 15], 15, 15, 15),
    ([10, 5, 15, 3, 7, None, 18], 1, 100, 58),
    ([10, 5, 15, 3, 7, None, 18], 11, 14, 0),
    ([10, 5, 15, 3, 7, None, 18], 16, 100, 18),
]
solvers = [range_sum_full, range_sum_collect, range_sum_pruned, range_sum_stack,
           range_sum_bfs, range_sum_inorder_stop, Solution().rangeSumBST]
for values, low, high, expected in tests:
    for f in solvers:
        got = f(build_tree(values), low, high)
        assert got == expected, (f.__name__, values, low, high, got, expected)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 8 test x 7 cách


In [18]:
import random

def bst_insert(root, val):
    # chèn lặp (không đệ quy) để dựng được cây rất sâu
    new = TreeNode(val)
    if root is None:
        return new
    cur = root
    while True:
        if val < cur.val:
            if cur.left is None:
                cur.left = new; return root
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = new; return root
            cur = cur.right

rng = random.Random(938)
for _ in range(300):
    vals = rng.sample(range(1, 200), rng.randint(1, 40))
    root = None
    for v in vals:
        root = bst_insert(root, v)
    low = rng.randint(1, 199); high = rng.randint(low, 199)
    expected = sum(v for v in vals if low <= v <= high)
    for f in solvers:
        assert f(root, low, high) == expected, (f.__name__, vals, low, high)
print("PASS 300 BST ngẫu nhiên")

PASS 300 BST ngẫu nhiên


In [19]:
# Cây lệch phải 20000 nút (giới hạn trên của đề): bản đệ quy có thể tràn, bản stack thì ổn
# Dựng trực tiếp chuỗi 1 -> 2 -> ... -> 20000 (mỗi nút chỉ có con phải): vẫn là BST hợp lệ
skew = TreeNode(1)
cur = skew
for v in range(2, 20001):
    cur.right = TreeNode(v)
    cur = cur.right
print("stack     :", range_sum_stack(skew, 1, 20000))
print("inorder   :", range_sum_inorder_stop(skew, 1, 20000))
try:
    print("đệ quy    :", range_sum_pruned(skew, 1, 20000))
except RecursionError as e:
    print("đệ quy    : RecursionError ->", e)

stack     : 200010000
inorder   : 200010000
đệ quy    : RecursionError -> maximum recursion depth exceeded


Lưu ý: LeetCode thực tế cho phép độ sâu đệ quy lớn hơn mặc định nên bản đệ quy thường vẫn **Accepted**, nhưng hiểu rủi ro này giúp bạn chọn bản lặp khi cần. (Nếu dựng cây lệch này bằng `bst_insert` thì mất `O(n²)` vì mỗi lần chèn phải đi hết chuỗi — nên ta nối trực tiếp.)

## 12. Lỗi người mới hay gặp

### Lỗi 1: Cắt tỉa sai chiều

In [20]:
def wrong_direction(node, low, high):
    if node is None:
        return 0
    if node.val < low:
        return wrong_direction(node.left, low, high)    # SAI: phải đi sang phải
    if node.val > high:
        return wrong_direction(node.right, low, high)   # SAI: phải đi sang trái
    return node.val + wrong_direction(node.left, low, high) + wrong_direction(node.right, low, high)

print("wrong_direction t2 [6,10] =", wrong_direction(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 10), "| đúng: 23")

wrong_direction t2 [6,10] = 10 | đúng: 23


**Vì sao sai:** `v < low` nghĩa là `v` **quá nhỏ**, giá trị lớn hơn nằm **bên phải**. **Mẹo nhớ:** "nhỏ quá thì sang phải (lớn hơn), lớn quá thì sang trái".

### Lỗi 2: Dùng khoảng mở `<` thay vì đoạn đóng `<=`

In [21]:
def wrong_open(node, low, high):
    if node is None:
        return 0
    here = node.val if low < node.val < high else 0
    return here + wrong_open(node.left, low, high) + wrong_open(node.right, low, high)

print("wrong_open t1 [7,15] =", wrong_open(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15), "| đúng: 32")

wrong_open t1 [7,15] = 10 | đúng: 32


### Lỗi 3: Quên base case `None` → `AttributeError`

In [22]:
def wrong_no_base(node, low, high):
    here = node.val if low <= node.val <= high else 0
    return here + wrong_no_base(node.left, low, high) + wrong_no_base(node.right, low, high)

try:
    wrong_no_base(TreeNode(5), 1, 10)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'val'


### Lỗi 4: Khi nút nằm trong đoạn lại chỉ đi một phía

Nếu `low <= v <= high`, cả hai cây con đều có thể còn giá trị hợp lệ. Chỉ đi một phía sẽ bỏ sót.

### Lỗi 5: Dùng biến toàn cục cộng dồn mà không reset giữa các test

Ví dụ `self.total = 0` đặt ở `__init__` nhưng LeetCode có thể gọi `rangeSumBST` nhiều lần trên cùng đối tượng → tổng cộng dồn sai. **Sửa:** dùng giá trị trả về của đệ quy, hoặc khởi tạo `total` ngay trong method.

In [23]:
class WrongAccum:
    def __init__(self):
        self.total = 0
    def rangeSumBST(self, root, low, high):
        if root:
            if low <= root.val <= high:
                self.total += root.val
            self.rangeSumBST(root.left, low, high)
            self.rangeSumBST(root.right, low, high)
        return self.total

w = WrongAccum()
print(w.rangeSumBST(build_tree([5]), 1, 10), w.rangeSumBST(build_tree([5]), 1, 10), "<- lần 2 phải là 5")

5 10 <- lần 2 phải là 5


## 13. Mẹo nhận dạng pattern

- Đề có **BST** + điều kiện theo **khoảng giá trị** (`low`, `high`, `> x`, `< x`) → **cắt tỉa**: so sánh với nút rồi chỉ đi về phía có thể còn đáp án.
- Bài cây "tính tổng / đếm / tìm" → **DFS đệ quy** với mẫu `if not node: return base; return combine(node, dfs(left), dfs(right))`.
- Cần thứ tự tăng dần trên BST → **inorder**.
- Python + cây có thể sâu `10^4` → cân nhắc **stack lặp**.

Bài liên quan: 700 Search in a BST, 669 Trim a BST, 270 Closest BST Value, 530 Minimum Absolute Difference in BST, 98 Validate BST, 404 Sum of Left Leaves.

## 14. Biến thể

### 14.1. Đếm số nút trong đoạn (thay vì tổng)

In [24]:
def range_count(node, low, high):
    if node is None:
        return 0
    if node.val < low:
        return range_count(node.right, low, high)
    if node.val > high:
        return range_count(node.left, low, high)
    return 1 + range_count(node.left, low, high) + range_count(node.right, low, high)

print(range_count(t2, 6, 10))   # 3

3


### 14.2. Liệt kê các giá trị trong đoạn theo thứ tự tăng

Inorder có cắt tỉa: chỉ đi trái khi `val > low`, chỉ đi phải khi `val < high`.

In [25]:
def range_values(node, low, high, out):
    if node is None:
        return out
    if node.val > low:
        range_values(node.left, low, high, out)
    if low <= node.val <= high:
        out.append(node.val)
    if node.val < high:
        range_values(node.right, low, high, out)
    return out

print(range_values(t2, 6, 13, []))   # [6, 7, 10, 13]

[6, 7, 10, 13]


### 14.3. Trim a BST (LeetCode 669) — cắt bỏ nút ngoài đoạn, trả cây mới

In [26]:
def trim_bst(node, low, high):
    if node is None:
        return None
    if node.val < low:
        return trim_bst(node.right, low, high)
    if node.val > high:
        return trim_bst(node.left, low, high)
    node.left = trim_bst(node.left, low, high)
    node.right = trim_bst(node.right, low, high)
    return node

print(to_list(trim_bst(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 13)))

[10, 7, 13, 6]


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Tính tay
Với cây t1 `[10,5,15,3,7,null,18]`, `low = 4`, `high = 12`: những nút nào được **thăm** bởi bản cắt tỉa? Tổng bằng bao nhiêu?

### Bài 2 — Tổng toàn cây (đệ quy)

In [27]:
def exercise_tree_sum(node) -> int:
    # TODO: base case None -> 0
    pass

# assert exercise_tree_sum(build_tree([10, 5, 15, 3, 7, None, 18])) == 58

### Bài 3 — Tự viết lời giải 938 có cắt tỉa

In [28]:
def exercise_range_sum(node, low: int, high: int) -> int:
    # TODO: 4 nhánh: None / < low / > high / trong đoạn
    pass

# assert exercise_range_sum(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15) == 32
# assert exercise_range_sum(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 10) == 23

### Bài 4 — Phiên bản lặp dùng stack

In [29]:
def exercise_range_sum_iterative(root, low: int, high: int) -> int:
    # TODO
    pass

# assert exercise_range_sum_iterative(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15) == 32

### Bài 5 — Search in a BST (LeetCode 700)

Trả về nút có giá trị `val` (hoặc `None`). Dùng tính chất BST để chỉ đi một nhánh.

In [30]:
def exercise_search_bst(root, val):
    # TODO
    pass

# assert exercise_search_bst(build_tree([4, 2, 7, 1, 3]), 2).val == 2
# assert exercise_search_bst(build_tree([4, 2, 7, 1, 3]), 5) is None

### Bài 6 — Mini challenge: Range Sum trên cây nhị phân **bất kỳ** và trên BST — so sánh số nút thăm

Viết hàm trả về cặp `(tổng, số_nút_thăm)` cho bản cắt tỉa. Sau đó trên cây cân bằng 1..1023, tìm đoạn `[low, high]` có độ rộng 10 khiến số nút thăm **lớn nhất**.

In [31]:
def exercise_sum_and_visits(node, low, high) -> tuple[int, int]:
    # TODO
    pass

# s, v = exercise_sum_and_visits(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15); assert s == 32

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | bắt đầu ở 10 (trong đoạn) | 5 trong đoạn → đi cả 3 và 7 | 15 > 12 → chỉ đi trái (None) |
| 2 | `val + trái + phải` | base case | `return node.val + f(left) + f(right)` |
| 3 | nhỏ quá → phải, lớn quá → trái | xem mục 5 | `return node.val + dfs(l) + dfs(r)` |
| 4 | stack thay đệ quy | `stack = [root]`, `pop()` | đẩy con theo 3 nhánh |
| 5 | so sánh `val` với nút | `while node and node.val != val` | `node = node.left if val < node.val else node.right` |
| 6 | trả tuple | cộng dồn từng thành phần | thử mọi `low` từ 1 tới 1014 |

## 17. Đáp án tham khảo

Bài 1: thăm `10, 5, 3, 7, 15` (5 nút; 18 không thăm vì 15 > 12 chỉ đi trái). Tổng = `10 + 5 + 7 = 22`.

In [32]:
print("visits:", visits_pruned(t1, 4, 12), "| tổng:", range_sum_pruned(t1, 4, 12))

visits: 5 | tổng: 22


In [33]:
def answer_tree_sum(node):
    if node is None:
        return 0
    return node.val + answer_tree_sum(node.left) + answer_tree_sum(node.right)

def answer_range_sum(node, low, high):
    if node is None:
        return 0
    if node.val < low:
        return answer_range_sum(node.right, low, high)
    if node.val > high:
        return answer_range_sum(node.left, low, high)
    return node.val + answer_range_sum(node.left, low, high) + answer_range_sum(node.right, low, high)

def answer_range_sum_iterative(root, low, high):
    total, stack = 0, [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if node.val < low:
            stack.append(node.right)
        elif node.val > high:
            stack.append(node.left)
        else:
            total += node.val
            stack.extend([node.left, node.right])
    return total

def answer_search_bst(root, val):
    node = root
    while node and node.val != val:
        node = node.left if val < node.val else node.right
    return node

def answer_sum_and_visits(node, low, high):
    if node is None:
        return 0, 0
    if node.val < low:
        s, v = answer_sum_and_visits(node.right, low, high)
        return s, v + 1
    if node.val > high:
        s, v = answer_sum_and_visits(node.left, low, high)
        return s, v + 1
    s1, v1 = answer_sum_and_visits(node.left, low, high)
    s2, v2 = answer_sum_and_visits(node.right, low, high)
    return node.val + s1 + s2, v1 + v2 + 1

assert answer_tree_sum(build_tree([10, 5, 15, 3, 7, None, 18])) == 58
assert answer_range_sum(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15) == 32
assert answer_range_sum(build_tree([10, 5, 15, 3, 7, 13, 18, 1, None, 6]), 6, 10) == 23
assert answer_range_sum_iterative(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15) == 32
assert answer_search_bst(build_tree([4, 2, 7, 1, 3]), 2).val == 2
assert answer_search_bst(build_tree([4, 2, 7, 1, 3]), 5) is None
s, v = answer_sum_and_visits(build_tree([10, 5, 15, 3, 7, None, 18]), 7, 15); assert s == 32
for values, low, high, expected in tests:
    assert answer_range_sum(build_tree(values), low, high) == expected
    assert answer_range_sum_iterative(build_tree(values), low, high) == expected

best = max(((answer_sum_and_visits(big, lo, lo + 9)[1], lo) for lo in range(1, 1015)))
print("Bài 6: đoạn rộng 10 thăm nhiều nút nhất:", f"[{best[1]}, {best[1] + 9}]", "với", best[0], "nút")
print("Đáp án tham khảo PASS toàn bộ test!")

Bài 6: đoạn rộng 10 thăm nhiều nút nhất: [512, 521] với 26 nút
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def rangeSumBST(self, root, low, high):
        if not root:
            return 0
        if root.val < low:                       # nhỏ quá -> sang phải
            return self.rangeSumBST(root.right, low, high)
        if root.val > high:                      # lớn quá -> sang trái
            return self.rangeSumBST(root.left, low, high)
        return (root.val + self.rangeSumBST(root.left, low, high)
                         + self.rangeSumBST(root.right, low, high))
```

**Câu thần chú:** *"Nhỏ quá sang phải, lớn quá sang trái, nằm giữa thì cộng và đi cả hai."*

**Checklist trước khi submit:**
- [ ] Base case `None → 0`.
- [ ] Đoạn **đóng** `[low, high]` (dùng `<=`).
- [ ] Cắt tỉa đúng chiều.
- [ ] Nút trong đoạn: đi **cả hai** phía.
- [ ] Không dùng biến tích lũy ở mức đối tượng mà không reset.
- [ ] Biết bản lặp bằng stack khi cây có thể rất sâu.